<a href="https://colab.research.google.com/github/milidshah/Provisional-Ballots-NC/blob/main/ncsbe_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗳️ NC Voting Information RAG Chatbot
**Source:** [NC State Board of Elections](https://www.ncsbe.gov/)

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot that:
- Scrapes relevant pages from `ncsbe.gov` (voting requirements, precincts, election dates, etc.)
- Indexes them into a local vector store (FAISS)
- Answers questions using an LLM with **conversational memory**

This model is supported by Groq


## Step 1: Install Dependencies

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-text-splitters \
    faiss-cpu \
    beautifulsoup4 \
    requests \
    sentence-transformers

print('✅ Dependencies installed.')

## Step 2: Configure API Keys
Set your API key below. The notebook supports OpenAI or Anthropic — uncomment the one you want to use.

In [ ]:
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

print('✅ Groq API key loaded.')

## Step 3: Scrape NCSBE Pages

We target the most relevant sections of the NCSBE website:
- Voting requirements & Voter ID
- How to register / who can register
- Upcoming elections & early voting
- Polling place / precinct data
- Vote by mail, provisional voting, military voting
- FAQs

In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
import time

# ── Target URLs ───────────────────────────────────────────────────────────────
NCSBE_URLS = [
    # Registering
    'https://www.ncsbe.gov/registering',
    'https://www.ncsbe.gov/registering/how-register',
    'https://www.ncsbe.gov/registering/who-can-register',
    'https://www.ncsbe.gov/registering/updating-registration',
    'https://www.ncsbe.gov/registering/checking-your-registration',
    'https://www.ncsbe.gov/registering/faq-voter-registration',
    # Voting
    'https://www.ncsbe.gov/voting',
    'https://www.ncsbe.gov/voting/upcoming-election',
    'https://www.ncsbe.gov/voting/voter-id',
    'https://www.ncsbe.gov/voting/vote-early-person',
    'https://www.ncsbe.gov/voting/vote-person-election-day',
    'https://www.ncsbe.gov/voting/vote-mail',
    'https://www.ncsbe.gov/voting/provisional-voting',
    'https://www.ncsbe.gov/voting/military-and-overseas-voting',
    'https://www.ncsbe.gov/voting/help-voters-disabilities',
    'https://www.ncsbe.gov/voting/voter-tools-and-forms',
    # Results & Data
    'https://www.ncsbe.gov/results-data/polling-place-data',
    # About Elections
    'https://www.ncsbe.gov/about-elections/types-elections',
    'https://www.ncsbe.gov/about-elections/county-boards-elections',
]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; NC-Voting-RAG-Bot/1.0)'
}

def scrape_page(url: str) -> Document | None:
    """Fetch a page and return a LangChain Document with cleaned text."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')

        # Remove nav, footer, scripts, and sidebar noise
        for tag in soup.select('nav, footer, script, style, .nav, .menu,\
                                .sidebar, .breadcrumb, header, [role=navigation]'):
            tag.decompose()

        # Get page title
        title = soup.find('h1')
        title_text = title.get_text(strip=True) if title else url

        # Get main content area (prefer <main> or article, fallback to body)
        main = soup.find('main') or soup.find('article') or soup.find('body')
        text = main.get_text(separator='\n', strip=True) if main else ''

        # Basic cleanup: collapse excessive blank lines
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        clean_text = '\n'.join(lines)

        if len(clean_text) < 100:
            print(f'  ⚠️  Skipped (too short): {url}')
            return None

        return Document(
            page_content=clean_text,
            metadata={'source': url, 'title': title_text}
        )
    except Exception as e:
        print(f'  ❌ Error fetching {url}: {e}')
        return None


print('🔍 Scraping NCSBE pages...')
raw_docs = []
for url in NCSBE_URLS:
    doc = scrape_page(url)
    if doc:
        raw_docs.append(doc)
        print(f'  ✅ {doc.metadata["title"][:60]}')
    time.sleep(0.5)  # polite crawl delay

print(f'\n📄 Scraped {len(raw_docs)} pages successfully.')

## Step 4: Split Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '.', ' ', ''],
)

chunks = splitter.split_documents(raw_docs)
print(f'✅ Split into {len(chunks)} chunks.')
print(f'   Sample chunk preview:\n{chunks[0].page_content[:300]}...')

## Step 5: Build the FAISS Vector Store

We use a local HuggingFace embedding model so no extra API calls are needed for embeddings.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('🔄 Loading embedding model (first run downloads ~90MB)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)

print('🔄 Building FAISS index...')
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save locally so you can reload without re-scraping
vectorstore.save_local('ncsbe_faiss_index')
print('✅ Vector store built and saved to ./ncsbe_faiss_index')

## Step 6: Build the Conversational RAG Chain

This chain:
1. Reformulates the user question with chat history context (so follow-ups work)
2. Retrieves relevant chunks from the vector store
3. Generates an answer grounded in retrieved context
4. Stores all turns in memory automatically

In [ ]:
!pip show langchain langchain-core langchain-community langchain-google-genai

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── 1. LLM ────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model='llama-3.3-70b-versatile',  # free, fast, very capable
    temperature=0,
    max_tokens=1024,
)

# ── 2. Retriever ──────────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5},
)

# ── 3. Contextualize question using chat history ──────────────────────────────
contextualize_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Given a chat history and the latest user question which might reference '
     'context from the chat history, formulate a standalone question that can '
     'be understood without the chat history. Do NOT answer it, just reformulate '
     'it if needed; otherwise return it as-is.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

contextualize_chain = contextualize_prompt | llm | StrOutputParser()

def contextualize_question(input: dict) -> str:
    if input.get('chat_history'):
        return contextualize_chain.invoke(input)
    return input['input']

# ── 4. QA prompt ──────────────────────────────────────────────────────────────
qa_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '''You are a helpful assistant specializing in North Carolina voting information.
Your answers are based on official information from the NC State Board of Elections (ncsbe.gov).

Guidelines:
- Answer only using the provided context. If the answer is not in the context, say so clearly.
- Be concise and accurate. For procedural questions (registration, ID requirements, dates), list steps clearly.
- Always note when voters should verify details directly at ncsbe.gov or contact their county board of elections,
  since election dates and precinct locations can change.
- Do not speculate or provide information not supported by the retrieved context.

Context:
{context}'''),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# ── 5. Full chain ─────────────────────────────────────────────────────────────
rag_chain = (
    RunnablePassthrough.assign(
        standalone_question=contextualize_question
    )
    | RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x['standalone_question'])),
        retrieved_docs=lambda x: retriever.invoke(x['standalone_question'])
    )
    | RunnablePassthrough.assign(
        answer=qa_prompt | llm | StrOutputParser()
    )
)

print('✅ Conversational RAG chain ready (Llama 3.3 70b via Groq).')

## Step 7: Chat Interface

Run this cell to start an interactive chat session. Type `quit` or `exit` to end.
The assistant remembers everything said earlier in the conversation.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

print('=' * 60)
print('🗳️  NC Voting Information Assistant')
print('   Powered by NCSBE.gov | Type "quit" to exit')
print('=' * 60)
print('Ask me about voter registration, Voter ID requirements,')
print('election dates, early voting, precinct locations, and more.\n')

while True:
    user_input = input('You: ').strip()

    if not user_input:
        continue
    if user_input.lower() in ('quit', 'exit', 'q'):
        print('\nGoodbye! Remember to vote! 🗳️')
        break

    try:
        result = rag_chain.invoke({
            'input': user_input,
            'chat_history': chat_history,
        })

        answer = result['answer']

        # Update memory
        chat_history.append(HumanMessage(content=user_input))
        chat_history.append(AIMessage(content=answer))

        # limit memory for efficiency
        chat_history = chat_history[-8:] # remember last 4 turns

        # Show sources
        sources = list({doc.metadata['source'] for doc in result.get('retrieved_docs', [])})

        print(f'\nAssistant 🤖: {answer}')
        if sources:
            print('\n📎 Sources:')
            for s in sources:
                print(f'   • {s}')
        print()

    except Exception as e:
        print(f'\n⚠️  Error: {e}\n')

## Step 8: Inspect Retrieved Chunks

debugging what gets retrieved for a given question.

In [ ]:
# test_query = 'What ID do I need to vote in North Carolina?'
# docs = retriever.invoke(test_query)
# for i, doc in enumerate(docs):
#     print(f'--- Chunk {i+1} | Source: {doc.metadata["source"]} ---')
#     print(doc.page_content[:400])
#     print()